In [15]:
import sys
from pathlib import Path
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# 1. Setup root path project dan pastikan direktori kerja aktif selalu di root
ROOT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT_DIR)
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

# 2. Konfigurasi Lingkungan GPU NVIDIA & Redam Warning Allocation
os.environ.pop("CUDA_VISIBLE_DEVICES", None)
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0"

import tensorflow as tf

# 3. Setup Eksplisit GPU Memory Growth (Cegah TF Menyerap Seluruh VRAM Sekaligus)
gpus = tf.config.list_physical_devices("GPU")
if gpus:
    for gpu in gpus:
        try:
            tf.config.experimental.set_memory_growth(gpu, True)
        except Exception as e:
            print(f"Warning memory growth: {e}")
    print(f"[Akselerasi GPU Aktif] Menggunakan perangkat: {gpus[0].name}")
else:
    print("[Perhatian] GPU tidak terdeteksi, berjalan pada mode CPU.")

from src.utils.config import Config
from src.utils.seed import set_seed
from src.preprocessing.agreement import AnnotatorAgreementAggregator
from src.preprocessing.normalizer import SlangNormalizer
from src.preprocessing.cleaner import TextCleaner
from src.preprocessing.tokenizer import TextTokenizer
from src.preprocessing.padder import SequencePadder
from src.preprocessing.topic_encoder import TopicEncoder
from src.evaluation.metrics import MetricCalculator
from src.evaluation.error_analysis import ErrorAnalyzer

# Inisialisasi konfigurasi parameter
cfg = Config()
set_seed(cfg.SEED)

print("=== KONFIGURASI PARAMETER (SESUAI DIAGRAM WORKFLOW) ===")
print(f"Project Root       : {ROOT_DIR}")
print(f"Random Seed        : {cfg.SEED}")
print(f"Vocab Size         : {cfg.VOCAB_SIZE:,} kata")
print(f"Max Sequence Len   : {cfg.MAX_LEN} token")
print(f"Text Embedding     : FastText cc.id.300 ({cfg.EMBEDDING_DIM}d)")
print(f"CNN Kernels/Filters: k={cfg.FILTER_SIZES}, filters={cfg.NUM_FILTERS} -> Text_Vec (384d)")
print(f"Topic Embedding    : {cfg.NUM_TOPICS} kategori kanonikal -> Topic_Vec ({cfg.TOPIC_EMBEDDING_DIM}d)")
print(f"Feature Fusion     : Concatenate ({cfg.NUM_FILTERS * len(cfg.FILTER_SIZES)} + {cfg.TOPIC_EMBEDDING_DIM} = 416d)")
print(f"Loss Function      : Binary Focal Loss (gamma={cfg.FOCAL_GAMMA}, alpha={cfg.FOCAL_ALPHA})")
print("Training Schedule  : 30 Epochs (5 Warmup + 20 Main + 5 Cooldown)")
print(f"Perangkat Komputasi: {gpus if gpus else 'CPU (Default)'}")


=== KONFIGURASI PARAMETER (SESUAI DIAGRAM WORKFLOW) ===
Project Root       : /home/josjiez/Documents/IndoToxic
Random Seed        : 42
Vocab Size         : 20,000 kata
Max Sequence Len   : 128 token
Text Embedding     : FastText cc.id.300 (300d)
CNN Kernels/Filters: k=[3, 4, 5], filters=128 -> Text_Vec (384d)
Topic Embedding    : 9 kategori kanonikal -> Topic_Vec (32d)
Feature Fusion     : Concatenate (384 + 32 = 416d)
Loss Function      : Binary Focal Loss (gamma=2.0, alpha=0.25)
Optimizer/Stopping : Adam (lr=0.001) + EarlyStopping (patience=5)


---
## 1. Source Data -> Data Curation -> Curated Data

- **Source Data**: IndoToxic2024 Raw Data (28,448 baris)
- **Data Curation Strategy**:
  1. Konsensus majority voting 29 anotator (threshold $\ge 0.5$) $\to$ Label 1 (Toxic) & Label 0 (Non-Toxic).
  2. **Toxic (3,995 baris)**: Seluruh sampel kelas minoritas toxic dipertahankan utuh 100% tanpa pengurangan.
  3. **Non-Toxic (10,000 baris)**: Diambil tepat 10,000 baris dengan *stratified sampling* berdasarkan kategori `topic` agar seluruh spektrum dan variansi topik terwakili secara proporsional.
- **Curated Data**: **13,995 baris** (3,995 Toxic + 10,000 Non-Toxic, rasio Toxic 28.55%).

In [16]:
from sklearn.model_selection import train_test_split

# 1. Muat Source Data (Raw 28,448 baris)
raw_path = ROOT_DIR / cfg.RAW_DATA_CSV
df_raw = pd.read_csv(raw_path)
print(f"[Source Data] Raw Data Loaded : {len(df_raw):,} baris x {df_raw.shape[1]} kolom")

# 2. Agregasi Majority Voting 29 Anotator
aggregator = AnnotatorAgreementAggregator(threshold=0.5)
df_processed = aggregator.process_dataframe(
    df_raw=df_raw,
    vote_col="toxicity",
    label_col="label",
    agreement_col="annotator_agreement",
)

# Bersihkan data bernilai null pada kolom teks
df_processed = df_processed.dropna(subset=["text"]).reset_index(drop=True)

# 3. Data Curation: Toxic 3,995 utuh + Non-Toxic 10,000 representatif variansi topik
toxic_df = df_processed[df_processed["label"] == 1].copy()
nontoxic_df = df_processed[df_processed["label"] == 0].copy()

TARGET_NONTOXIC = 10000

# Stratified sampling pada Non-Toxic berdasarkan kategori topik
# Topik langka (hanya 1 sampel) dipertahankan langsung agar stratifikasi valid
topic_counts = nontoxic_df["topic"].value_counts()
rare_topics = topic_counts[topic_counts == 1].index
rare_df = nontoxic_df[nontoxic_df["topic"].isin(rare_topics)]
common_df = nontoxic_df[~nontoxic_df["topic"].isin(rare_topics)]

n_needed = TARGET_NONTOXIC - len(rare_df)
sampled_common, _ = train_test_split(
    common_df,
    train_size=n_needed,
    stratify=common_df["topic"],
    random_state=cfg.SEED,
)

curated_nontoxic = pd.concat([sampled_common, rare_df]).reset_index(drop=True)

# Gabungkan Toxic (3,995) dan Non-Toxic (10,000) -> 13,995 baris curated data
df_curated = pd.concat([toxic_df, curated_nontoxic]).sample(frac=1.0, random_state=cfg.SEED).reset_index(drop=True)

print("\n=== HASIL DATA CURATION INDOTOXIC2024 ===")
print(f"Kelas Toxic (Dipertahankan 100%) : {len(toxic_df):,} baris")
print(f"Kelas Non-Toxic (Sampled Variansi) : {len(curated_nontoxic):,} baris")
print(f"[Curated Data] Total Dataset     : {len(df_curated):,} baris")
print(f"Rasio Kelas Toxic                : {df_curated['label'].mean():.2%}")

print("\nDistribusi Topik Non-Toxic (Asli vs Sampled 10.000):")
top_topics = nontoxic_df["topic"].value_counts().head(6).index
for t in top_topics:
    p_orig = (nontoxic_df["topic"] == t).mean()
    p_samp = (curated_nontoxic["topic"] == t).mean()
    print(f"  - {t:<15}: Asli {p_orig:.2%} -> Sampled {p_samp:.2%}")

[Source Data] Raw Data Loaded : 28,448 baris x 14 kolom

=== HASIL DATA CURATION INDOTOXIC2024 ===
Kelas Toxic (Dipertahankan 100%) : 3,995 baris
Kelas Non-Toxic (Sampled Variansi) : 10,000 baris
[Curated Data] Total Dataset     : 13,995 baris
Rasio Kelas Toxic                : 28.55%

Distribusi Topik Non-Toxic (Asli vs Sampled 10.000):
  - Jewish         : Asli 28.96% -> Sampled 28.94%
  - Disabilitas    : Asli 24.91% -> Sampled 24.89%
  - Tionghoa       : Asli 20.63% -> Sampled 20.62%
  - UNKNOWN        : Asli 8.08% -> Sampled 8.08%
  - Rohingya       : Asli 5.64% -> Sampled 5.64%
  - Kristen        : Asli 5.12% -> Sampled 5.11%


---
## 2. Split Data: Stratified Sampling

Pembagian dataset kurasi (13,995 baris) secara berstrata (*stratified*) berdasarkan label dengan determinisme `SEED=42`:
- **Train (70%)**: **9,796** baris (Toxic: 2,796 | Non-Toxic: 7,000)
- **Validation (15%)**: **2,099** baris (Toxic: 599 | Non-Toxic: 1,500)
- **Test (15%)**: **2,100** baris (Toxic: 600 | Non-Toxic: 1,500)  
*(Total: 9,796 + 2,099 + 2,100 = 13,995 baris)*

In [17]:
from sklearn.model_selection import train_test_split

# Split data: 70% Train, 30% (Val + Test)
train_df, temp_df = train_test_split(
    df_curated,
    test_size=0.30,
    stratify=df_curated["label"],
    random_state=cfg.SEED,
)

# Split sisa 30% menjadi 15% Val dan 15% Test
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    stratify=temp_df["label"],
    random_state=cfg.SEED,
)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print("=== PEMBAGIAN PARTISI STRATIFIED (SESUAI DIAGRAM) ===")
print(f"TRAIN (70%)        : {len(train_df):,} baris (Toxic: {(train_df['label']==1).sum():,}, Non-Toxic: {(train_df['label']==0).sum():,})")
print(f"VALIDATION (15%)   : {len(val_df):,} baris (Toxic: {(val_df['label']==1).sum():,}, Non-Toxic: {(val_df['label']==0).sum():,})")
print(f"TEST (15%)         : {len(test_df):,} baris (Toxic: {(test_df['label']==1).sum():,}, Non-Toxic: {(test_df['label']==0).sum():,})")
print(f"TOTAL              : {len(train_df) + len(val_df) + len(test_df):,} baris")

=== PEMBAGIAN PARTISI STRATIFIED (SESUAI DIAGRAM) ===
TRAIN (70%)        : 9,796 baris (Toxic: 2,796, Non-Toxic: 7,000)
VALIDATION (15%)   : 2,099 baris (Toxic: 599, Non-Toxic: 1,500)
TEST (15%)         : 2,100 baris (Toxic: 600, Non-Toxic: 1,500)
TOTAL              : 13,995 baris


---
## 3. Preprocessing (Train & Val/Test Pipeline Identik)

Aturan preprocessing teks sesuai spesifikasi metodologi penelitian:
1. **Lowercasing (Case folding)**: Menyeragamkan seluruh huruf ke huruf kecil.
2. **Leet-speak decoding**: Mengubah angka/simbol penyamar makian (`4nj1ng` -> `anjing`, `@` -> `a`, `3` -> `e`, dsb.).
3. **Regex sanitization**: Hapus URL web, mention `@user`, tag HTML `<...>`, dan karakter non-alfanumerik.
4. **Reduksi huruf berulang**: Kompresi perulangan emosional (`begooolll` -> `begool`).
5. **Kamus slang (Alay dictionary)**: Padanan bahasa gaul medsos ke kata baku (`gw` -> `saya`, `gak` -> `tidak`).
6. **(No stemming)**: Stemming dihindari agar imbuhan pembawa konteks emosional tidak rusak.
7. **Stopword TIDAK dihapus**: Kata negasi (`tidak`, `bukan`) dan kata tugas penting dipertahankan agar n-gram CNN membaca konteks utuh.

In [18]:
import sys
from pathlib import Path
import os

# Set root path project dan ubah direktori kerja ke root
ROOT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT_DIR)

if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

In [19]:
# Inisialisasi normalizer leet-speak & slang serta text cleaner
normalizer = SlangNormalizer(config=cfg)
cleaner = TextCleaner()

def preprocess_pipeline(texts: list[str]) -> list[str]:
    """
    Pipeline Preprocessing Teks Medsos:
    - Lowercasing
    - Leet-speak decoding
    - Regex sanitization (URL, Mention, HTML, punctuation)
    - Reduksi huruf berulang
    - Kamus slang
    - No stemming
    - Stopword TIDAK dihapus (dipertahankan utuh untuk n-gram CNN)
    """
    cleaned = []
    for t in texts:
        # 1 & 2 & 4 & 5. Normalizer menangani leet-speak decoding, huruf berulang, dan kamus slang
        t_norm = normalizer.normalize(str(t))
        # 3. Cleaner menangani regex URL, mention, tag HTML, lowercase, dan sanitasi tanda baca
        t_clean = cleaner.clean(t_norm)
        cleaned.append(t_clean)
    return cleaned

# Eksekusi preprocessing pada masing-masing partisi
train_df["text_preprocessed"] = preprocess_pipeline(train_df["text"].tolist())
val_df["text_preprocessed"] = preprocess_pipeline(val_df["text"].tolist())
test_df["text_preprocessed"] = preprocess_pipeline(test_df["text"].tolist())

# Tampilkan sampel hasil transformasi
sample_idx = 0
print("Contoh Hasil Preprocessing:")
print(f"Teks Asli        : {train_df['text'].iloc[sample_idx]}")
print(f"Hasil Preprocess : {train_df['text_preprocessed'].iloc[sample_idx]}")

[2026-09-30 11:01:51] [INFO] [SlangNormalizer]: Leksikon termuat: 270 slang, 186 emoji.
Contoh Hasil Preprocessing:
Teks Asli        : MasyaAllah Alhamdulillah B 🤲 Mari kita simak bersama
Hasil Preprocess : masyaallah alhamdulillah b mari kita simak bersama


---
## 4. Tokenizer & Transform (Batas Anti-Leakage)

Parameter & Aturan Tokenisasi:
- **Word-level tokenization**: Pemecahan kalimat berbasis kata.
- **`VOCAB_SIZE = 20,000`**: 20.000 kata paling sering muncul + token `<PAD>` (0) dan `<OOV>` (1).
- **`MAX_LEN = 128`**: Panjang sekuens seragam (post-padding dan post-truncation).
- **Prinsip Anti-Leakage**: `Tokenizer.fit()` **HANYA** pada data Train (70%).
- **`Tokenizer.transform()`**: Menerapkan tokenizer hasil fit Train pada Validation dan Test.
- **Topic Encoding**: 7 kategori kanonikal (`Politik`, `Agama`, `SARA`, `Gender`, `Pemilu2024`, `Umum`, `UNKNOWN`).

In [20]:
# 1. Inisialisasi Tokenizer & Fit HANYA pada Train data
tokenizer = TextTokenizer(
    vocab_size=cfg.VOCAB_SIZE,
    oov_token=cfg.OOV_TOKEN,
    pad_token=cfg.PAD_TOKEN,
)
tokenizer.fit(train_df["text_preprocessed"].tolist())

# 2. Transform teks ke sekuens integer token
train_seq = tokenizer.texts_to_sequences(train_df["text_preprocessed"].tolist())
val_seq = tokenizer.texts_to_sequences(val_df["text_preprocessed"].tolist())
test_seq = tokenizer.texts_to_sequences(test_df["text_preprocessed"].tolist())

# 3. Padding ke panjang tetap MAX_LEN = 128 token
padder = SequencePadder(max_len=cfg.MAX_LEN, padding="post", truncating="post")
X_text_train = padder.pad(train_seq)
X_text_val = padder.pad(val_seq)
X_text_test = padder.pad(test_seq)

# 4. Topic Encoding 7 kategori kanonikal (integer ID 0..6)
topic_encoder = TopicEncoder(canonical_topics=cfg.CANONICAL_TOPICS)
X_topic_train = topic_encoder.transform(train_df["topic"]).reshape(-1, 1)
X_topic_val = topic_encoder.transform(val_df["topic"]).reshape(-1, 1)
X_topic_test = topic_encoder.transform(test_df["topic"]).reshape(-1, 1)

# Target label biner
y_train = train_df["label"].to_numpy(dtype=np.int32)
y_val = val_df["label"].to_numpy(dtype=np.int32)
y_test = test_df["label"].to_numpy(dtype=np.int32)

print("=== BENTUK INPUT MODEL (TENSOR SHAPES) ===")
print(f"X_text_train  : {X_text_train.shape}  (vocab={len(tokenizer.word_index):,} kata terdaftar)")
print(f"X_topic_train : {X_topic_train.shape}  (7 kategori: 0..6)")
print(f"X_text_val    : {X_text_val.shape}")
print(f"X_topic_val   : {X_topic_val.shape}")
print(f"X_text_test   : {X_text_test.shape}")
print(f"X_topic_test  : {X_topic_test.shape}")

=== BENTUK INPUT MODEL (TENSOR SHAPES) ===
X_text_train  : (9796, 128)  (vocab=20,000 kata terdaftar)
X_topic_train : (9796, 1)  (7 kategori: 0..6)
X_text_val    : (2099, 128)
X_topic_val   : (2099, 1)
X_text_test   : (2100, 128)
X_topic_test  : (2100, 1)


---
## 5. Model Architecture & Model Training

### Arsitektur: Context-Aware Dual-Input Multi-Kernel CNN
1. **Text Branch (FastText cc.id.300 + Multi-Kernel CNN)**:
   - Input Teks: (128)
   - FastText Embedding: (128, 300)
   - 3x Paralel Conv1D: kernel $k \in \{3, 4, 5\}$ masing-masing 128 filter (ReLU)
   - GlobalMaxPooling1D per kernel $\to$ Concatenate: **`Text_Vec (384d)`**
2. **Topic Branch (Topic Embedding)**:
   - Input Topik: (1)
   - Categorical Embedding (7 topik $\to$ 32d) $\to$ **`Topic_Vec (32d)`**
3. **Multi-Modal Feature Fusion**:
   - Concatenate: `[Text_Vec (384d) || Topic_Vec (32d)]` = **`416d`**
   - Dropout(0.5) $\to$ Dense(128, ReLU) $\to$ Dropout(0.3)
   - Fully Connected Output: **Dense(1, Sigmoid)** $\to$ Probabilitas Toxic [0..1]
4. **Training Strategy**:
   - Loss: **Binary Focal Loss** ($\gamma = 2.0, \alpha = 0.25$)
   - Optimizer: **Adam** ($lr = 0.001$)
   - **EarlyStopping**: Memantau `val_macro_f1` (patience = 5)

In [21]:
# ==============================================================================
# 5. MODEL ARCHITECTURE & TRAINING EKSPLISIT (VANILLA SINGLE-KERNEL CNN)
# ==============================================================================
import tensorflow as tf
from tensorflow.keras import layers, models, optimizers
import numpy as np

# 1. Parameter Model & Training Eksplisit
VOCAB_SIZE = int(cfg.VOCAB_SIZE)          # 20,000
MAX_LEN = int(cfg.MAX_LEN)                # 128
EMBEDDING_DIM = int(cfg.EMBEDDING_DIM)    # 300
KERNEL_SIZE = 3                           # Single Kernel k=3 (Vanilla Baseline)
NUM_FILTERS = 128                         # 128 filter
NUM_TOPICS = len(cfg.CANONICAL_TOPICS)    # 9 kategori
TOPIC_EMBEDDING_DIM = 32                  # 32d
DENSE_UNITS = 128                         # Hidden layer units
DROPOUT_FUSION = 0.5                      # Dropout setelah fusion
DROPOUT_DENSE = 0.3                       # Dropout sebelum output

BATCH_SIZE = 32
TOTAL_EPOCHS = 30
WARMUP_EPOCHS = 5
MAIN_EPOCHS = 20
COOLDOWN_EPOCHS = 5
BASE_LR = float(cfg.LEARNING_RATE)        # 0.001

# 2. Bangun Topologi Graf Model Secara Eksplisit (Single-Kernel Vanilla CNN)
print(">>> [1/4] Membangun Arsitektur Single-Kernel Vanilla CNN secara eksplisit...")

# --- Stream 1: Teks Sequence Input ---
text_input = layers.Input(shape=(MAX_LEN,), dtype="int32", name="text_input")
text_emb = layers.Embedding(
    input_dim=VOCAB_SIZE,
    output_dim=EMBEDDING_DIM,
    name="trainable_embedding"
)(text_input)

conv = layers.Conv1D(
    filters=NUM_FILTERS,
    kernel_size=KERNEL_SIZE,
    activation="relu",
    padding="valid",
    name="conv1d_single"
)(text_emb)
text_vec = layers.GlobalMaxPooling1D(name="gmp_single")(conv)  # 128d

# --- Stream 2: Topic Integer Input ---
topic_input = layers.Input(shape=(1,), dtype="int32", name="topic_input")
topic_emb = layers.Embedding(
    input_dim=NUM_TOPICS,
    output_dim=TOPIC_EMBEDDING_DIM,
    name="topic_embedding"
)(topic_input)
topic_vec = layers.Flatten(name="topic_vec")(topic_emb)        # 32d

# --- Multi-Modal Fusion & Dense Classification Head ---
fusion = layers.Concatenate(name="fusion_vec")([text_vec, topic_vec])  # 160d (128 + 32)
x = layers.Dropout(DROPOUT_FUSION, name="dropout_fusion")(fusion)
x = layers.Dense(DENSE_UNITS, activation="relu", name="dense_hidden")(x)
x = layers.Dropout(DROPOUT_DENSE, name="dropout_dense")(x)
output_prob = layers.Dense(1, activation="sigmoid", name="output_prob")(x)

model = models.Model(
    inputs=[text_input, topic_input],
    outputs=output_prob,
    name="Vanilla_Single_Kernel_CNN"
)

# 3. Definisi Fungsi Binary Focal Loss Eksplisit
print(">>> [2/4] Mengompilasi dengan Binary Focal Loss (gamma=2.0, alpha=0.25)...")
def binary_focal_loss(y_true, y_pred, gamma=2.0, alpha=0.25):
    """Binary Focal Loss penyeimbang kelas minoritas toxic."""
    epsilon = tf.keras.backend.epsilon()
    y_pred = tf.clip_by_value(y_pred, epsilon, 1.0 - epsilon)
    y_true = tf.cast(y_true, tf.float32)
    
    p_t = y_true * y_pred + (1.0 - y_true) * (1.0 - y_pred)
    alpha_t = y_true * alpha + (1.0 - y_true) * (1.0 - alpha)
    focal_weight = alpha_t * tf.pow(1.0 - p_t, gamma)
    bce = -tf.math.log(p_t)
    return tf.reduce_mean(focal_weight * bce)

model.compile(
    optimizer=optimizers.Adam(learning_rate=BASE_LR),
    loss=binary_focal_loss,
    metrics=["accuracy"],
    jit_compile=False  # Nonaktifkan XLA rematerialization untuk stabilitas VRAM GPU 6GB
)

# 4. Learning Rate Scheduler 3-Fase Eksplisit
print(">>> [3/4] Menyiapkan Tri-Stage LR Scheduler (5 Warmup + 20 Main + 5 Cooldown)...")
def tri_stage_lr_schedule(epoch, lr):
    if epoch < WARMUP_EPOCHS:
        return BASE_LR * float(epoch + 1) / float(WARMUP_EPOCHS)
    elif epoch < (WARMUP_EPOCHS + MAIN_EPOCHS):
        return BASE_LR
    else:
        step = epoch - (WARMUP_EPOCHS + MAIN_EPOCHS) + 1
        return BASE_LR * (0.5 ** step)

lr_callback = tf.keras.callbacks.LearningRateScheduler(tri_stage_lr_schedule, verbose=1)

# 5. Eksekusi Training Standalone Langsung
print(f"\n>>> [4/4] Memulai Fitting Model: {TOTAL_EPOCHS} Epochs penuh pada {len(X_text_train):,} train & {len(X_text_val):,} val...")
print("          (EarlyStopping dibypass total agar konvergensi 30 epoch tuntas)\n")

raw_history = model.fit(
    [X_text_train, X_topic_train],
    y_train,
    validation_data=([X_text_val, X_topic_val], y_val),
    batch_size=BATCH_SIZE,
    epochs=TOTAL_EPOCHS,
    callbacks=[lr_callback],
    verbose=1
)
history = raw_history.history
print("\n=== Pelatihan 30 Epochs Selesai 100%! ===")


Model: "Context_Aware_Dual_Input_CNN"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ text_input          │ (None, 128)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ trainable_embedding │ (None, 128, 300)  │  6,000,000 │ text_input[0][0]  │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_k3 (Conv1D)  │ (None, 126, 128)  │    115,328 │ trainable_embedd… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_k4 (Conv1D)  │ (None, 125, 128)  │    153,728 │ trainable_embedd… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_k5 (Conv1D)  │ (None, 124, 128)  │    192,128 │ trainable_embedd… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ topic_input         │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ gmp_k3              │ (None, 128)       │          0 │ conv1d_k3[0][0]   │
│ (GlobalMaxPooling1… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ gmp_k4              │ (None, 128)       │          0 │ conv1d_k4[0][0]   │
│ (GlobalMaxPooling1… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ gmp_k5              │ (None, 128)       │          0 │ conv1d_k5[0][0]   │
│ (GlobalMaxPooling1… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ topic_embedding     │ (None, 1, 32)     │        288 │ topic_input[0][0] │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ text_vec            │ (None, 384)       │          0 │ gmp_k3[0][0],     │
│ (Concatenate)       │                   │            │ gmp_k4[0][0],     │
│                     │                   │            │ gmp_k5[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ topic_vec (Flatten) │ (None, 32)        │          0 │ topic_embedding[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ fusion_vec          │ (None, 416)       │          0 │ text_vec[0][0],   │
│ (Concatenate)       │                   │            │ topic_vec[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_fusion      │ (None, 416)       │          0 │ fusion_vec[0][0]  │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_hidden        │ (None, 128)       │     53,376 │ dropout_fusion[0… │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_dense       │ (None, 128)       │          0 │ dense_hidden[0][… │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ output_prob (Dense) │ (None, 1)         │        129 │ dropout_dense[0]… │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 6,514,977 (24.85 MB)

 Trainable params: 6,514,977 (24.85 MB)

 Non-trainable params: 0 (0.00 B)

Memulai training dengan 3500 sampel train dan 700 sampel val...
[2026-09-30 11:01:53] [INFO] [ModelTrainer]: Memulai proses fitting model...
[2026-09-30 11:01:53] [INFO] [ModelTrainer]: Focal Loss diaktifkan (gamma=2.0, alpha=0.25)
Epoch 1/20
110/110 ━━━━━━━━━━━━━━━━━━━━ 9s 71ms/step - accuracy: 0.7194 - loss: 0.0588 - val_accuracy: 0.7143 - val_loss: 0.0528
Epoch 2/20
110/110 ━━━━━━━━━━━━━━━━━━━━ 8s 75ms/step - accuracy: 0.7669 - loss: 0.0405 - val_accuracy: 0.7657 - val_loss: 0.0547
Epoch 3/20
110/110 ━━━━━━━━━━━━━━━━━━━━ 8s 70ms/step - accuracy: 0.9177 - loss: 0.0181 - val_accuracy: 0.7657 - val_loss: 0.0806
Epoch 4/20
110/110 ━━━━━━━━━━━━━━━━━━━━ 8s 70ms/step - accuracy: 0.9709 - loss: 0.0085 - val_accuracy: 0.7214 - val_loss: 0.1275
Epoch 5/20
110/110 ━━━━━━━━━━━━━━━━━━━━ 8s 69ms/step - accuracy: 0.9834 - loss: 0.0068 - val_accuracy: 0.7429 - val_loss: 0.1084
Epoch 6/20
110/110 ━━━━━━━━━━━━━━━━━━━━ 7s 68ms/step - accuracy: 0.9920 - loss: 0.0037 - val_accuracy: 0.7557 - val_loss: 0

---
## 6. Prediction (Output) & Model Evaluation

Evaluasi komprehensif pada Test Set (2,800 baris out-of-sample) sesuai diagram alur:
1. **Prediction (Output)**: Klasifikasi probabilitas $\to$ label biner **Toxic** (1) vs **Non-Toxic** (0) pada threshold 0.5
2. **Metrik Evaluasi**:
   - **Macro-F1** (Target utama $\ge 0.78$)
   - **Precision**
   - **Recall (Toxic)** (Target $\ge 0.75$)
   - **AUC-ROC**
   - **Confusion Matrix**
   - **Analisis FP/FN** (False Positive & False Negative Error Analysis)

In [22]:
# 1. Inferensi Prediksi pada Test Set
y_test_subset = y_test
test_df_subset = test_df.copy()

y_prob = model.predict([X_text_test, X_topic_test], batch_size=32, verbose=0).flatten() if hasattr(model, 'predict') and not hasattr(model, 'predict_proba') else model.predict_proba(X_text_test, X_topic_test).flatten()
y_pred = (y_prob >= 0.5).astype(int)

# 2. Kalkulasi Metrik Lengkap via MetricCalculator
calc = MetricCalculator()
metrics = calc.compute_all(y_true=y_test_subset, y_pred=y_pred, y_prob=y_prob)

print("================ HASIL EVALUASI MODEL TEST SET ================")
print(f"Macro-F1 Score    : {metrics['macro_f1']:.4f}  (Target riset >= 0.78)")
print(f"Macro-Precision   : {metrics['precision_macro']:.4f}")
print(f"Recall (Toxic)    : {metrics['per_class']['toxic']['recall']:.4f}  (Target riset >= 0.75)")
print(f"AUC-ROC           : {metrics['auc_roc']:.4f}")
print(f"Accuracy          : {metrics['accuracy']:.4f}")
print(f"Per-Class Toxic F1: {metrics['per_class']['toxic']['f1']:.4f}")
print(f"Confusion Matrix  : {metrics['confusion_matrix']}")
print("===============================================================")

# 3. Analisis FP / FN (Error Analysis)
test_df_subset["pred_label"] = y_pred
test_df_subset["pred_prob"] = y_prob

analyzer = ErrorAnalyzer()
error_res = analyzer.analyze(
    df=test_df_subset,
    text_col="text",
    true_label_col="label",
    pred_label_col="pred_label",
    prob_col="pred_prob",
    top_n=3,
)

print(f"\nTotal Evaluasi Sample : {error_res['total_samples']}")
print(f"False Positives (FP)  : {error_res['false_positives_count']} kasus (Label Asli Non-Toxic, Ditebak Toxic)")
print(f"False Negatives (FN)  : {error_res['false_negatives_count']} kasus (Label Asli Toxic, Ditebak Non-Toxic)")

print("\n--- Top False Positives (FP) ---")
for item in error_res["top_false_positives"]:
    print(f"Prob: {item['pred_prob']:.3f} | Teks: {item['text'][:90]}...")

print("\n--- Top False Negatives (FN) ---")
for item in error_res["top_false_negatives"]:
    print(f"Prob: {item['pred_prob']:.3f} | Teks: {item['text'][:90]}...")

================ HASIL EVALUASI MODEL TEST SET ================
Macro-F1 Score    : 0.4186  (Target riset >= 0.78)
Macro-Precision   : 0.3600
Recall (Toxic)    : 0.0000  (Target riset >= 0.75)
AUC-ROC           : 0.7414
Accuracy          : 0.7200
Per-Class Toxic F1: 0.0000
Confusion Matrix  : [[432, 0], [168, 0]]

Total Evaluasi Sample : 600
False Positives (FP)  : 0 kasus (Label Asli Non-Toxic, Ditebak Toxic)
False Negatives (FN)  : 168 kasus (Label Asli Toxic, Ditebak Non-Toxic)

--- Top False Positives (FP) ---

--- Top False Negatives (FN) ---
Prob: 0.222 | Teks: BWF Super 1000 V VICTOR CHINAOPEN ICTOR BADMINTALK China Open 2023 Women's Doubles Quarter...
Prob: 0.235 | Teks: Victor China Open 2023 32 Besar: XD: Rehan Naufal Kusharjanto/ Lisa Ayu Kusumawati 🇮🇩 vs T...
Prob: 0.247 | Teks: China Open 2023: 5 Wakil Indonesia Melaju ke Babak 16 Besar #parboaboa #chinaopen2023 #bul...


---
## 7. Pengumpulan & Ekspor Data Kasus Kesalahan Prediksi (FP & FN)

Mengumpulkan seluruh data yang salah diprediksi oleh model ke dalam Dataframe khusus untuk dianalisis pola kelemahannya:
- **False Positive (FP)**: Teks sebenarnya **Non-Toxic (0)**, tetapi model salah menuduh sebagai **Toxic (1)** dengan keyakinan probabilitas tinggi.
- **False Negative (FN)**: Teks sebenarnya **Toxic (1)**, tetapi model gagal mendeteksi dan menganggapnya **Non-Toxic (0)** (*paling berbahaya/risiko lolos sensor*).

Seluruh baris FP dan FN disimpan ke file CSV di `outputs/error_analysis/` agar mudah diaudit oleh tim.

In [23]:
# Pisahkan data False Positive (FP) dan False Negative (FN) secara lengkap
df_fp = test_df_subset[(test_df_subset["label"] == 0) & (test_df_subset["pred_label"] == 1)].copy()
df_fn = test_df_subset[(test_df_subset["label"] == 1) & (test_df_subset["pred_label"] == 0)].copy()

# Urutkan berdasarkan tingkat keparahan error (probabilitas tertinggi untuk FP, terendah untuk FN)
df_fp = df_fp.sort_values(by="pred_prob", ascending=False).reset_index(drop=True)
df_fn = df_fn.sort_values(by="pred_prob", ascending=True).reset_index(drop=True)

print("=== REKAPITULASI SAMPEL SALAH PREDIKSI (ERROR AUDIT) ===")
print(f"Total Test Evaluated : {len(test_df_subset):,} baris")
print(f"Total False Positive : {len(df_fp):,} baris (Non-Toxic dituduh Toxic)")
print(f"Total False Negative : {len(df_fn):,} baris (Toxic gagal terdeteksi)")

# Tampilkan 5 sampel teratas False Positive (FP)
print("\n" + "="*45 + " 5 CONTOH FALSE POSITIVE (FP) " + "="*45)
for i, row in df_fp.head(5).iterrows():
    print(f"[{i+1}] Topik: {row['topic']} | Pred Prob: {row['pred_prob']:.4f}")
    print(f"    Teks Asli   : {row['text']}")
    print(f"    Preprocessed: {row['text_preprocessed']}")
    print("-" * 100)

# Tampilkan 5 sampel teratas False Negative (FN)
print("\n" + "="*45 + " 5 CONTOH FALSE NEGATIVE (FN) " + "="*45)
for i, row in df_fn.head(5).iterrows():
    print(f"[{i+1}] Topik: {row['topic']} | Pred Prob: {row['pred_prob']:.4f}")
    print(f"    Teks Asli   : {row['text']}")
    print(f"    Preprocessed: {row['text_preprocessed']}")
    print("-" * 100)

# Simpan seluruh baris FP dan FN ke folder outputs/error_analysis/
out_dir = ROOT_DIR / "outputs" / "error_analysis"
out_dir.mkdir(parents=True, exist_ok=True)

cols_to_export = ["text_id", "topic", "label", "pred_label", "pred_prob", "text", "text_preprocessed"]
available_cols = [c for c in cols_to_export if c in test_df_subset.columns]

fp_path = out_dir / "false_positives.csv"
fn_path = out_dir / "false_negatives.csv"
combined_error_path = out_dir / "all_misclassified.csv"

df_fp[available_cols].to_csv(fp_path, index=False)
df_fn[available_cols].to_csv(fn_path, index=False)

# File gabungan dengan penanda tipe error
df_fp["error_type"] = "False Positive"
df_fn["error_type"] = "False Negative"
pd.concat([df_fp, df_fn])[available_cols + ["error_type"]].to_csv(combined_error_path, index=False)

print(f"\nSemua data FP dan FN berhasil diekspor ke:")
print(f"  1. {fp_path}")
print(f"  2. {fn_path}")
print(f"  3. {combined_error_path}")

=== REKAPITULASI SAMPEL SALAH PREDIKSI (ERROR AUDIT) ===
Total Test Evaluated : 600 baris
Total False Positive : 0 baris (Non-Toxic dituduh Toxic)
Total False Negative : 168 baris (Toxic gagal terdeteksi)

============================================= 5 CONTOH FALSE POSITIVE (FP) =============================================

============================================= 5 CONTOH FALSE NEGATIVE (FN) =============================================
[1] Topik: Tionghoa | Pred Prob: 0.2220
    Teks Asli   : BWF Super 1000 V VICTOR CHINAOPEN ICTOR BADMINTALK China Open 2023 Women's Doubles Quarter Final Apriyani Rahayu Siti Fadia Silva Ramadhanti Baek Ha Na [2] Lee So Hee  15 21 10 21 18 21 Photo: Badmintalk Results Update of China Open 2023 - QF

Apri/Fadia harus akui kehebatan unggulan kedua. Tetap semangat! 

Foto: Badmintalk 

#china #chinaopen2023 #badmintalk #badmintoneurope #juara #champion #winner #brilliant #amazing #incredible #awesome #badminton #badmintonplayer #love #indonesia #b